```python
from json import loads
import numpy as np

def f():
    global m
    m = {"。": 0}
    for i in "train", "valid":
        for i in open(f"../tmp/news2016zh_{i}.json"):
            i = loads(i)
            for i in i["keywords"], i["desc"], i["title"], i["source"], i["content"]:
                for i in i:
                    yield m.setdefault(i, len(m))
                if "\u4e00" <= i <= "\u9fff":
                    yield 0

a = np.fromiter(f(), "H")
a.tofile("../tmp/news2016zh2.H")
open("../tmp/news2016zh2.txt", "w").write("".join(sorted(m, key=m.__getitem__)))
```

```python
import numpy as np
from multiprocessing.pool import ThreadPool

def f(i):
    c = np.zeros(21482, "I")
    np.add.at(c, a[len(a) * i // 12 : len(a) * (i + 1) // 12], 1)
    return c

a = np.memmap(f"../tmp/news2016zh.H", "H")
with ThreadPool(12) as p:
    p = np.sum(p.map(f, range(12)), 0)

t @ np.log2(t) #-9.655227239498798
    
t = p > p.max() // 100
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh100.H")
open("../tmp/news2016zh100.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])

t = p > p.max() // 1000
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh1000.H")
open("../tmp/news2016zh1000.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])
```

```python
print(np.array([open("../tmp/news2016zh100.txt").read()]).view("U1")[np.fromfile('../tmp/news2016zh100.H','H',10000)].view('U10000')[0])
```

```python
m = np.array([open("../tmp/news2016zh.txt").read()]).view("I")
p *= (0x4e00 <= m) & (m <= 0x9fff)
t = p > p.max() // 100
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zhMin100.H")
open("../tmp/news2016zhMin100.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])
```

```python
def f(m):
    if isinstance(m, Sequential):
        for i in range(len(m)):
            if isinstance(m[i], BatchNorm1dFlat):
                t = BatchNorm1dFlat(32)
                t.weight = m[i].weight
                t.bias = m[i].bias
                m[i] = t
m = Learner(
    DataLoaders(Dat(256, 256, 10), Dat(256, 256, 1000)),
    m.model.apply(f).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(8)),
    cbs=[ShowGraphCallback, SaveModelCallback(fname='model3b', every_epoch=1, with_opt=1, reset_on_fit=0)],
)
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(*x.shape[:-2], -1, s.k)


class Affine(Module):
    def __init__(s, k):
        super().__init__()
        s.w = Parameter(torch.ones(k))
        s.b = Parameter(torch.zeros(k))

    def forward(s, x):
        return s.b.addcmul(x, s.w)


class LayerNorm(Module):
    @staticmethod
    def forward(x):
        return F.layer_norm(x, x.shape[1:], None, None, 1e-5)


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        n = int(np.log2(x.size(1)))
        t = []
        for _ in range(n):
            x = s[0](x)
            t.append(x)
        t = list(s[1](torch.cat(t, 1)).split([t.size(1) for t in t], 1))
        x = None
        while t:
            x = s[2](t.pop() if x is None else torch.cat([t.pop(), x], 1))
        return x


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def __init__(s, g):
        super().__init__()
        s.g = g

    def forward(s, x, t):
        x = x.view(len(x), s.g, -1, x.size(-1))
        return -x.log_softmax(3)[
            torch.arange(len(x), device="cuda")[:, None],
            :,
            torch.arange(x.size(2), device="cuda"),
            t,
        ].mean()


def Accuracy(g):
    for i in range(g):

        def Accuracy(x, t, i=i):
            return accuracy(x.view(len(x), g, -1, x.size(-1))[:, i], t)

        Accuracy.__name__ = str(int(i))
        yield Accuracy

In [ ]:
# m = Learner(
#     DataLoaders(Dat(256, 256, 100), Dat(256, 256, 1000)),
#     Sequential(
#         Embedding(684, 32),
#         PyramidAutoEncoder(
#             Sequential(
#                 View(64), Linear(64, 32, 0), InstanceNorm1d(0), Affine(32), Swish()
#             ),
#             Sequential(
#                 Linear(32, 32, 0),
#                 InstanceNorm1d(0),
#                 Affine(32),
#                 Swish(),
#                 Linear(32, 32),
#             ),
#             Sequential(
#                 InstanceNorm1d(0), Affine(32), Swish(), Linear(32, 64, 0), View(32)
#             ),
#         ),
#         Linear(32, 32, 0),
#         InstanceNorm1d(0),
#         Affine(32),
#         Swish(),
#         Linear(32, 684),
#     ).cuda(),
#     MultiCrossEntropyLoss(8),
#     metrics=map(partial(Accuracy, 8), range(8)),
#     cbs=[
#         ShowGraphCallback,
#         SaveModelCallback(fname="model4", with_opt=1, reset_on_fit=0),
#     ],
# )
m = Learner(
    DataLoaders(Dat(256, 256, 10), Dat(256, 256, 1000)),
    Sequential(
        Embedding(684, 32),
        PyramidAutoEncoder(
            Sequential(View(64), Linear(64, 32), LayerNorm(), Swish()),
            Sequential(
                Linear(32, 32),
                LayerNorm(),
                Swish(),
                Linear(32, 32),
            ),
            Sequential(LayerNorm(), Swish(), Linear(32, 64), View(32)),
        ),
        Linear(32, 32),
        LayerNorm(),
        Swish(),
        Linear(32, 684),
    ).cuda(),
    MultiCrossEntropyLoss(8),
    metrics=Accuracy(8),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model4b", with_opt=1, reset_on_fit=0),
    ],
)
m.load("model4", "cuda")
m.validate()
# m.lr_find()

In [ ]:
x = m.dls.a[0]
print(m.dls.m[x].view('U256')[0])

In [ ]:
for i in m.dls.m[m.model(x.to('cuda',torch.long)[None]).argmax(2).view(8,256).cpu()].view('U256')[:,0]:
    print(i, end='\n====================\n')

In [ ]:
m.fine_tune(1000)